In [1]:
import pandas as pd

FILE_NAME = "products_details_final_clean.csv"
# Đọc file CSV
try:
    df = pd.read_csv(
        FILE_NAME,
        sep=',',
        engine='python',
        on_bad_lines='skip'
    )
except Exception as e:
    print(f"Lỗi khi tải file {FILE_NAME}: {e}")
    raise

category_counts = df['category'].value_counts()

# Xuất ra file TXT
output_path = "category_counts.txt"
with open(output_path, "w", encoding="utf-8") as f:
    for category, count in category_counts.items():
        f.write(f"{category} - {count}\n")

print("Xuất file thành công tại:", output_path)


Xuất file thành công tại: category_counts.txt


In [2]:
import pandas as pd
import numpy as np

def load_categories_from_file(filename):
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            categories = [line.strip() for line in f if line.strip()]
        print(f"    -> Đã đọc {len(categories)} mục từ file {filename}")
        return categories
    except FileNotFoundError:
        print(f"    -> LỖI: Không tìm thấy file {filename}")
        return []
    except Exception as e:
        print(f"    -> LỖI khi đọc file {filename}: {e}")
        return []


try:
    df = pd.read_csv('products_details_final_clean.csv', on_bad_lines='skip')

    df['discount_percent'] = df['discount_percent'].astype(str).str.replace('%', '', regex=False)
    df['discount_percent'] = pd.to_numeric(df['discount_percent'], errors='coerce')

    df['store_name'] = df['store_name'].fillna('Unknown')

    if 'link' in df.columns:
        df = df.drop(columns=['link'])

    print("Đọc và làm sạch cơ bản thành công.")

    MIN_COUNT_THRESHOLD = 50
    print(f"\nNgưỡng số lượng danh mục để giữ riêng: {MIN_COUNT_THRESHOLD}")

    print("\nĐang tải 4 file danh mục nhóm chính...")

    fashion_cats = load_categories_from_file('unique_fashion.txt')
    food_cats = load_categories_from_file('unique_food.txt')
    cosmetics_cats = load_categories_from_file('unique_cosmetics.txt')
    electronics_cats = load_categories_from_file('unique_electronics.txt')

    category_counts = df['category'].value_counts()
    sufficient_cats = category_counts[category_counts >= MIN_COUNT_THRESHOLD].index.tolist()

    print(f"\nCó {len(sufficient_cats)} danh mục đủ lớn (>= {MIN_COUNT_THRESHOLD}):")
    print(sufficient_cats)

    print("\nBắt đầu gộp nhóm...")

    conditions = [
        df['category'].isin(sufficient_cats),
        df['category'].isin(fashion_cats),
        df['category'].isin(food_cats),
        df['category'].isin(cosmetics_cats),
        df['category'].isin(electronics_cats)
    ]

    choices = [
        df['category'],
        'Thời trang khác',
        'Đồ ăn khác',
        'Mỹ phẩm khác',
        'Đồ điện tử khác'
    ]

    df['category_group'] = np.select(conditions, choices, default='Khác')

    print("\n--- Kết quả gộp nhóm ---")
    print(df['category_group'].value_counts())
    print("\nTổng số danh mục sau gộp:", df['category_group'].nunique())

    print("\n5 dòng đầu:")
    print(df[['category', 'category_group', 'price']].head())

    # --- 5. Xuất file CSV đã xử lý ---
    output_csv_name = 'product_detail_processing.csv'
    df.to_csv(output_csv_name, index=False, encoding='utf-8')

    print(f"\nĐã xuất file CSV: {output_csv_name}")

    # --- 6. Xuất file TXT chứa danh mục sau khi gộp ---
    output_txt_name = 'unique_category_groups.txt'
    unique_groups = sorted(df['category_group'].unique())

    with open(output_txt_name, 'w', encoding='utf-8') as f:
        for cat in unique_groups:
            f.write(cat + '\n')

    print(f"\nĐã xuất file TXT nhóm danh mục: {output_txt_name}")
    print("Danh mục sau gộp:", unique_groups)

    # --- 7. Xuất file TXT chứa danh mục + số lượng ---
    group_counts_txt = 'category_groups_with_counts.txt'
    group_counts = df['category_group'].value_counts()

    with open(group_counts_txt, 'w', encoding='utf-8') as f:
        for cat, count in group_counts.items():
            f.write(f"{cat} - {count}\n")

    print(f"\nĐã xuất file TXT danh mục kèm số lượng: {group_counts_txt}")
    print(group_counts)

except Exception as e:
    print(f"Đã xảy ra lỗi: {e}")


Đọc và làm sạch cơ bản thành công.

Ngưỡng số lượng danh mục để giữ riêng: 50

Đang tải 4 file danh mục nhóm chính...
    -> Đã đọc 42 mục từ file unique_fashion.txt
    -> Đã đọc 18 mục từ file unique_food.txt
    -> Đã đọc 12 mục từ file unique_cosmetics.txt
    -> Đã đọc 77 mục từ file unique_electronics.txt

Có 20 danh mục đủ lớn (>= 50):
['Snacks & Confectionery', 'Food Staples & Cooking Essentials', 'Set bộ đi chơi', 'Pet Food', 'Nguyên liệu nấu ăn & làm bánh', 'Đồ ăn nhẹ & bánh kẹo', 'Trang điểm', 'Milk Formula & Baby Food', 'Tableware', 'Kitchen Appliances', 'Dụng cụ chăm sóc Sắc đẹp', 'Áo', 'Đồ ngủ nữ', 'Feeding Essentials', 'Vali và Túi du lịch đa năng', 'Chăm sóc da mặt', 'Kitchenware', 'Quần áo trẻ em', 'Building Toys', 'Chăm sóc cá nhân']

Bắt đầu gộp nhóm...

--- Kết quả gộp nhóm ---
category_group
Snacks & Confectionery               1156
Food Staples & Cooking Essentials     736
Set bộ đi chơi                        573
Pet Food                              478
Nguyên l